# 03 Generate and build pairs
Attach: repo zip dataset, 01_sft output, both 02_dpo outputs. Accelerator: GPU T4 x2. Internet: on.

In [ ]:
import glob, os, shutil, zipfile
matches = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
if not matches:
    for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(archive).extractall("/tmp/unzipped")
    matches = glob.glob("/tmp/unzipped/**/submission_adapter.py", recursive=True)
assert matches, "attach the repo zip as a Kaggle dataset"
REPO = "/tmp/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.dirname(matches[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(REPO))

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "transformers", transformers.__version__, "datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())

In [ ]:
def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}/config.json", recursive=True))
    assert hits, f"no checkpoint matching {pattern} under /kaggle/input"
    return os.path.dirname(hits[0])

SFT = find("sft/best")
DPO_HH = find("dpo_hh/best")
DPO_UF = find("dpo_ultrafeedback/best")
GEN_ARGS = "--max_new_tokens 256 --batch_size 8"
ALPACA = "eval_sets/alpaca_eval_300_seed42.json"
HH = "eval_sets/hh_rlhf_eval_300_seed42.json"
os.makedirs(f"{OUT}/gen", exist_ok=True)
os.makedirs(f"{OUT}/eval", exist_ok=True)

## Sanity check on 3 prompts

In [ ]:
!python scripts/generate.py --model {SFT} --prompts_path {ALPACA} --output_path /tmp/sanity.json --generator sft --dataset alpaca_eval --limit 3 {GEN_ARGS}
import json
for row in json.load(open("/tmp/sanity.json")):
    print(row["instruction"], "\n->", row["output"], "\n")

## Baseline (base model) and SFT

In [ ]:
!python scripts/generate.py --model Qwen/Qwen2.5-0.5B --prompts_path {ALPACA} --output_path {OUT}/gen/base_alpaca.json --generator qwen2.5-0.5b-base --dataset alpaca_eval {GEN_ARGS}
!python scripts/generate.py --model {SFT} --prompts_path {ALPACA} --output_path {OUT}/gen/sft_alpaca.json --generator sft --dataset alpaca_eval {GEN_ARGS}
!python scripts/generate.py --model {SFT} --prompts_path {HH} --output_path {OUT}/gen/sft_hh.json --generator sft --dataset hh_rlhf_style_eval {GEN_ARGS}

## DPO models

In [ ]:
!python scripts/generate.py --model {DPO_UF} --prompts_path {ALPACA} --output_path {OUT}/gen/dpo_ultrafeedback_alpaca.json --generator dpo_ultrafeedback --dataset alpaca_eval {GEN_ARGS}
!python scripts/generate.py --model {DPO_HH} --prompts_path {HH} --output_path {OUT}/gen/dpo_hh_hh.json --generator dpo_hh --dataset hh_rlhf_style_eval {GEN_ARGS}

## Pairwise files

In [ ]:
!python scripts/build_pairs.py --subject_path {OUT}/gen/dpo_ultrafeedback_alpaca.json --baseline_path {OUT}/gen/sft_alpaca.json --prompts_path {ALPACA} --kind general --output_path {OUT}/eval/dpo_vs_sft_alpaca_pairs.jsonl
!python scripts/build_pairs.py --subject_path {OUT}/gen/dpo_hh_hh.json --baseline_path {OUT}/gen/sft_hh.json --prompts_path {HH} --kind safety --output_path {OUT}/eval/dpo_vs_sft_hh_pairs.jsonl

In [ ]:
!cd /kaggle/working && zip -rq generation_outputs.zip outputs/gen outputs/eval
!ls -lh /kaggle/working